# Experiment 3 — Video Object Segmentation via Label Propagation (T05 refactor)

**Protocol:** non-parametric label propagation (Jabri et al. 2020, DINO context design), no training, frame-0 GT given. Video Swin-T per frame (zero-shot for AnyUp3D) → upsampler → 56² features → `davis.propagate_labels` → official DAVIS-2017 semi-supervised J&F (`metrics.eval_clip_jf`, first and last frame not scored, per-object aggregation).
**Conditions:** `bilinear`, `anyup2d` (per frame), `anyup3d`.

Built on the shared modules; see `docs/plan/P1_experiments/T05_exp2_exp3_refactor.md`.

- **New protocol** (`LEGACY = False`): 30 val clips, all frames, Swin-T Kinetics normalization, chunks of 16 (no overlap: the backbone is per frame), `PROP_SIZE=56`.
- **`LEGACY = True`** (temporary regression check): first 2 clips, every 12th frame, ImageNet normalization, bilinear + AnyUp3D, and the old Exp 3 functions re-run inline on the **same backbone chunk**. PASS = upsampled features identical (max_abs = 0) and propagated masks identical (both on CPU, as the old code). Old vs official metric reported as INFO.

No feature cache. Masks are written locally as palette PNGs (official results layout) and zipped to Drive; per-clip CSV + summary JSON go to `<OUTPUT_ROOT>/<RUN_NAME>/`. T4 is enough. **Runtime → Restart session** before running if `anyup` was imported before.

In [ ]:
# ── Cell 1 · Repo + Drive + deps ────────────────────────────────────────────
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "experiments-task02"   # T05 continues on the T02 branch
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    # existing clone may be on another branch — switch to REPO_BRANCH and update it
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
if "anyup" in sys.modules:
    raise RuntimeError("anyup was already imported in this runtime — Runtime → Restart session, then run again")
subprocess.run(["pip", "install", "-e", REPO_DIR, "-q"], check=True)
subprocess.run(["pip", "install", "transformers", "timm", "scikit-image", "scipy", "-q"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2 · Config ─────────────────────────────────────────────────────────
LEGACY       = False   # ← True = regression check vs the old Exp 3 code (temporary, removed later)
N_CLIPS      = None    # None = all 30 val clips; int = first N (smoke test). LEGACY forces 2
CKPT_ANYUP3D = "/content/drive/MyDrive/results/anyup3d_step6500.pth"   # step6000 vs 6500 still open (plan.md)
OUTPUT_ROOT  = "/content/drive/MyDrive/results/T05/exp3"
MASK_LOCAL   = "/content/t05_exp3_masks"   # PNGs written here (fast), zipped to Drive at the end
DAVIS_DEST   = "/content"
DEVICE       = "cuda"
SEED         = 0

CONDITIONS   = ["bilinear", "anyup2d", "anyup3d"]
BASELINES    = ["bilinear", "anyup2d"]   # paired_stats: anyup3d vs each of these (if run)
SWIN_NORM    = "kinetics"                # Swin-T weights' own input stats (T01 default)

PROP_SIZE    = 56      # ↓ guide = upsampler output = propagation grid. Affinity per frame = PROP_SIZE² × (1+N_LAST)·PROP_SIZE² floats
                       #   (56: ≈ 275 MB). Dependent: N_LAST, Q_CHUNK, GPU memory of the clip features (N·768·PROP_SIZE²·4 B, ≤ 1 GB at 56)
CHUNK_SIZE   = 16      # ↓ frames per backbone / AnyUp3D call → peak memory and AnyUp3D temporal context. Dependent: Q_CHUNK, PROP_SIZE
OVERLAP      = 0       # Swin-T per frame: backbone-only overlap would not change any feature (Q9b). Must be < CHUNK_SIZE
Q_CHUNK      = None    # ↓ AnyUp3D query chunk (None = no chunking, as old Exp 3 at 56²). Dependent: CHUNK_SIZE, PROP_SIZE
TAU, TOP_K   = 0.07, 7 # propagation temperature / neighbours (Jabri et al.; DINO)
N_LAST       = 6       # ↓ context frames besides frame 0 → affinity memory. Dependent: PROP_SIZE

RUN_VISUALS    = False # optional stills + video (Cell 8)
N_VISUAL_CLIPS = 4     # ↓ rows of the stills figure
VISUAL_FPS     = 4
VISUAL_SIZE    = 224   # display size of the figure panels (independent of PROP_SIZE)

FRAME_STEP   = 12      # legacy: every 12th frame (old Exp 3)

if LEGACY:
    N_CLIPS    = 2
    CONDITIONS = ["bilinear", "anyup3d"]   # old Exp 3 conditions
    SWIN_NORM  = "imagenet"                # old Exp 3 frame_transform

RUN_NAME = f"{'legacy' if LEGACY else 'new'}_p{PROP_SIZE}_c{CHUNK_SIZE}_o{OVERLAP}_q{Q_CHUNK}_{SWIN_NORM}"
OUT_DIR  = f"{OUTPUT_ROOT}/{RUN_NAME}"
MASK_ROOT = f"{MASK_LOCAL}/{RUN_NAME}"     # <MASK_ROOT>/<condition>/<clip>/<frame stem>.png
print(f"LEGACY={LEGACY} RUN_NAME={RUN_NAME} CONDITIONS={CONDITIONS}")

In [ ]:
# ── Cell 3 · Imports, data, models ─────────────────────────────────────────
import random, time, shutil
from pathlib import Path
import numpy as np
import cv2
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

from anyup import davis, metrics
from anyup.backbones import BackboneLoader, BackboneOutput
from anyup.video import TensorSource, iter_backbone_chunks
from anyup.upsamplers import UpsamplerLoader

def set_seed(seed):
    """Fixed seeds (the pipeline has no random step; recorded for reproducibility)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

RESULTS = []   # legacy checks: (name, passed: bool | None for INFO, detail)
def record(name, passed, detail):
    RESULTS.append((name, passed, detail))
    tag = "INFO" if passed is None else ("PASS" if passed else "FAIL")
    print(f"[{tag}] {name}: {detail}")

set_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0))

ROOT  = davis.download_davis(DAVIS_DEST)
CLIPS = davis.list_clips(ROOT, "val")
if N_CLIPS is not None:
    CLIPS = CLIPS[:N_CLIPS]
print(f"{len(CLIPS)} clips: {CLIPS}")

bb  = BackboneLoader.load("swin_t", device=DEVICE, guide_size=PROP_SIZE, temporal_mode="per_frame", norm=SWIN_NORM)
UPS = {}
if "bilinear" in CONDITIONS:
    UPS["bilinear"] = UpsamplerLoader.load("bilinear", device=DEVICE)
if "anyup2d" in CONDITIONS:
    UPS["anyup2d"] = UpsamplerLoader.load("anyup2d", device=DEVICE)
if "anyup3d" in CONDITIONS:
    UPS["anyup3d"] = UpsamplerLoader.load("anyup3d", ckpt_path=CKPT_ANYUP3D, device=DEVICE, q_chunk_size=Q_CHUNK)
print("upsamplers:", list(UPS))
shutil.rmtree(MASK_ROOT, ignore_errors=True)   # this run's masks only (no stale files)

In [ ]:
# ── Cell 4 · LEGACY only: old Exp 3 code (verbatim, comments trimmed) ──────
# Old Cell 2 ANYUP3D_ARGS + Cell 4 model load + Cell 6 upsample_3d / upsample_bilinear + Cell 7 propagation
# + Cell 8 compute_J / compute_F. Changes (T05 log):
#   - AnyUp3D state dict from the safe loader, not torch.load(weights_only=False)
#   - inputs: the T01 chunk (tensor resize, ImageNet-normalized guide) instead of PIL + raw [0,1] guide (Q20)
#   - no float16 feature cache between upsampling and propagation
if LEGACY:
    from anyup.model import AnyUp
    from anyup.upsamplers import load_anyup3d_checkpoint
    device = torch.device(DEVICE)
    T_CHUNK = 100
    N_LAST_FRAMES = N_LAST

    ANYUP3D_ARGS = dict(
        input_dim       = 3,
        qk_dim          = 128,
        kernel_size     = 1,
        kernel_size_lfu = 5,
        window_ratio    = 0.1,
        num_heads       = 4,
        t_k             = 3,
        window_t        = 4,
    )
    model_3d = AnyUp(**ANYUP3D_ARGS).to(device).eval()
    sd   = load_anyup3d_checkpoint(CKPT_ANYUP3D)["anyup"]
    missing, unexpected = model_3d.load_state_dict(sd, strict=False)
    record("old AnyUp3D load (strict=False)", not missing and not unexpected,
           f"missing={len(missing)} unexpected={len(unexpected)}")
    for p in model_3d.parameters():
        p.requires_grad_(False)

    @torch.no_grad()
    def upsample_3d(feats_TCHW, guides_T3HW_raw):
        T = feats_TCHW.shape[0]
        out_chunks = []
        for start in range(0, T, T_CHUNK):
            end = min(start + T_CHUNK, T)
            feat_chunk  = feats_TCHW[start:end].unsqueeze(0).permute(0, 2, 1, 3, 4).to(device)
            guide_chunk = guides_T3HW_raw[start:end].unsqueeze(0).permute(0, 2, 1, 3, 4).to(device)
            out = model_3d(guide_chunk, feat_chunk)          # (1, 768, chunk, PROP_SIZE, PROP_SIZE)
            out_chunks.append(out.squeeze(0).permute(1, 0, 2, 3).cpu())
            del feat_chunk, guide_chunk, out
            torch.cuda.empty_cache()
        return torch.cat(out_chunks, dim=0)   # (T, 768, PROP_SIZE, PROP_SIZE)

    @torch.no_grad()
    def upsample_bilinear(feats_TCHW):
        return F.interpolate(
            feats_TCHW,
            size=(PROP_SIZE, PROP_SIZE),
            mode="bilinear",
            align_corners=False
        )

    def load_ann(ann_path):
        return np.array(Image.open(ann_path))

    def build_label_maps_frame0(ann0_HW, n_objects, H, W):
        ann_resized = cv2.resize(
            ann0_HW, (W, H), interpolation=cv2.INTER_NEAREST
        ).astype(np.int64)
        n_ch    = n_objects + 1
        N       = H * W
        label   = np.zeros((n_ch, N), dtype=np.float32)
        ann_flat = ann_resized.flatten()
        for obj_id in range(n_ch):
            label[obj_id, ann_flat == obj_id] = 1.0
        return torch.from_numpy(label)

    @torch.no_grad()
    def propagate_clip(feats_TCHW, ann_paths, sampled_idxs):
        T, C, H, W = feats_TCHW.shape
        N = H * W
        feats_flat = feats_TCHW.view(T, C, N).permute(0, 2, 1)
        feats_flat = F.normalize(feats_flat, dim=-1)
        ann0 = load_ann(ann_paths[sampled_idxs[0]])
        n_objects = int(ann0.max())
        n_ch = n_objects + 1
        label_maps = torch.zeros(T, n_ch, N)
        label_maps[0] = build_label_maps_frame0(ann0, n_objects, H, W)
        for t in range(1, T):
            past      = list(range(max(1, t - N_LAST_FRAMES), t))
            context_t = [0] + past
            ctx_feats  = feats_flat[context_t].reshape(-1, C)
            ctx_labels = label_maps[context_t].permute(1, 0, 2).reshape(n_ch, -1)
            q = feats_flat[t]
            A = torch.mm(q, ctx_feats.T) / TAU
            topk_vals, topk_idx = A.topk(TOP_K, dim=-1)
            A_sparse = torch.full_like(A, float("-inf"))
            A_sparse.scatter_(1, topk_idx, topk_vals)
            A_soft = F.softmax(A_sparse, dim=-1)
            new_labels = torch.mm(A_soft, ctx_labels.T)
            label_maps[t] = new_labels.T
        pred = label_maps.argmax(dim=1)
        return pred.view(T, H, W)

    def compute_J(pred_HW, gt_HW):
        obj_ids = np.unique(gt_HW)
        obj_ids = obj_ids[obj_ids > 0]
        if len(obj_ids) == 0:
            return 1.0
        j_scores = []
        for obj_id in obj_ids:
            pred_fg = (pred_HW == obj_id)
            gt_fg   = (gt_HW   == obj_id)
            inter   = np.logical_and(pred_fg, gt_fg).sum()
            union   = np.logical_or(pred_fg,  gt_fg).sum()
            j_scores.append(inter / union if union > 0 else 1.0)
        return float(np.mean(j_scores))

    def mask_to_boundary(mask_HW, dilation_px=2):
        mask_u8 = mask_HW.astype(np.uint8)
        kernel  = np.ones((dilation_px * 2 + 1, dilation_px * 2 + 1), np.uint8)
        eroded  = cv2.erode(mask_u8, kernel, iterations=1)
        return mask_u8 - eroded

    def compute_F(pred_HW, gt_HW, bound_th=0.008, dilation_px=2):
        obj_ids = np.unique(gt_HW)
        obj_ids = obj_ids[obj_ids > 0]
        if len(obj_ids) == 0:
            return 1.0
        bound_pix = max(1, int(np.ceil(bound_th * np.linalg.norm(gt_HW.shape))))
        struct = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                            (2 * bound_pix + 1, 2 * bound_pix + 1))
        f_scores = []
        for obj_id in obj_ids:
            pred_b = mask_to_boundary((pred_HW == obj_id).astype(np.uint8), dilation_px)
            gt_b   = mask_to_boundary((gt_HW   == obj_id).astype(np.uint8), dilation_px)
            n_pred, n_gt = pred_b.sum(), gt_b.sum()
            if n_pred == 0 and n_gt == 0:
                f_scores.append(1.0); continue
            if n_pred == 0 or n_gt == 0:
                f_scores.append(0.0); continue
            pred_dil = cv2.dilate(pred_b, struct)
            gt_dil   = cv2.dilate(gt_b,   struct)
            precision = np.logical_and(pred_b, gt_dil).sum() / (n_pred + 1e-8)
            recall    = np.logical_and(gt_b,   pred_dil).sum() / (n_gt   + 1e-8)
            f_scores.append(float(2 * precision * recall / (precision + recall + 1e-8)))
        return float(np.mean(f_scores))

    BIL_CPU = UpsamplerLoader.load("bilinear", device="cpu")   # old upsample_bilinear ran on CPU (T03: GPU ≠ CPU at ~1e-6)
    OLD_METRIC = {c: {"J": [], "F": []} for c in CONDITIONS}  # old frame-pooled J/F (frame 0 included), INFO

    def legacy_compare(clip, cond, chunks, feats_gpu, pred_gpu, ap, fp, idx, gt):
        assert len(chunks) == 1, f"legacy expects one chunk, got {len(chunks)}"
        ch = chunks[0]
        f_cpu, g_cpu = ch.features.cpu(), ch.guide.cpu()
        if cond == "anyup3d":
            old_feats = upsample_3d(f_cpu, g_cpu)                                    # GPU model, CPU out
            new_feats = feats_gpu.cpu()
        else:
            old_feats = upsample_bilinear(f_cpu)                                     # CPU
            new_feats = BIL_CPU(BackboneOutput(f_cpu, g_cpu, dict(ch.meta)), output_size=PROP_SIZE).features
            record(f"{clip} bilinear GPU vs CPU", None,
                   f"max_abs={(feats_gpu.cpu() - new_feats).abs().max().item():.2e}")
        record(f"{clip} {cond}: new == old features", torch.equal(new_feats, old_feats),
               f"shape={tuple(new_feats.shape)} max_abs={(new_feats - old_feats).abs().max().item():.3e}")
        old_pred = propagate_clip(old_feats, ap, idx)                                # CPU
        new_pred = davis.propagate_labels(new_feats, davis.load_annotation(ap[idx[0]]),
                                          tau=TAU, top_k=TOP_K, n_last=N_LAST)       # CPU
        record(f"{clip} {cond}: new == old masks (CPU)", torch.equal(new_pred, old_pred),
               f"shape={tuple(new_pred.shape)} differing_px={(new_pred != old_pred).sum().item()}")
        record(f"{clip} {cond}: main-loop GPU masks vs CPU", None,
               f"differing_px={(pred_gpu != new_pred).sum().item()} / {new_pred.numel()}")
        for t, i in enumerate(idx):                                                  # old metric on the scored PNGs
            pred = np.array(Image.open(Path(MASK_ROOT, cond, clip, f"{fp[i].stem}.png")))
            OLD_METRIC[cond]["J"].append(compute_J(pred, gt[t]))
            OLD_METRIC[cond]["F"].append(compute_F(pred, gt[t]))
else:
    print("LEGACY=False — skipped")

In [ ]:
# ── Cell 5 · Main loop ──────────────────────────────────────────────────────
PER_CLIP = {c: {} for c in CONDITIONS}   # condition → clip → eval_clip_jf result
rows, frame_checks = [], []
t_start = time.perf_counter()
for ci, clip in enumerate(CLIPS):
    src_full = davis.open_clip(ROOT, clip)
    fp, ap   = davis.frame_paths(ROOT, clip), davis.ann_paths(ROOT, clip)
    idx      = list(range(0, len(src_full), FRAME_STEP)) if LEGACY else list(range(len(src_full)))
    src      = TensorSource(src_full.get(idx)) if LEGACY else src_full   # (T, 3, 480, W) [0, 1]
    gt       = davis.load_annotations(ROOT, clip)[idx]                     # (T, 480, W) uint8, native
    ann0     = davis.load_annotation(ap[idx[0]])                           # frame-0 GT, native

    chunks = list(iter_backbone_chunks(bb, src, chunk_size=CHUNK_SIZE, overlap=OVERLAP))   # Swin-T once per clip
    fidx = [f for ch in chunks for f in ch.meta["frame_index"]]
    frame_checks.append(fidx == list(range(len(idx))))

    for cond in CONDITIONS:
        t0 = time.perf_counter()
        feats = torch.cat([UPS[cond](ch, output_size=PROP_SIZE).features for ch in chunks])   # (T, 768, P, P) GPU
        pred  = davis.propagate_labels(feats, ann0, tau=TAU, top_k=TOP_K, n_last=N_LAST)      # (T, P, P) CPU
        mdir = Path(MASK_ROOT, cond, clip)
        for t, i in enumerate(idx):
            davis.save_mask_png(pred[t], ap[0], mdir / f"{fp[i].stem}.png")                  # native res + palette
        pred_native = np.stack([np.array(Image.open(mdir / f"{fp[i].stem}.png")) for i in idx])   # what the toolkit reads
        res = metrics.eval_clip_jf(gt, pred_native)
        PER_CLIP[cond][clip] = res
        j, f = float(np.mean(res["J_M"])), float(np.mean(res["F_M"]))
        rows.append(dict(clip=clip, condition=cond, J=j, F=f, JF=(j + f) / 2, n_objects=res["n_objects"],
                         n_frames=len(idx), n_scored=res["n_scored"], n_chunks=len(chunks),
                         time_s=round(time.perf_counter() - t0, 2)))
        if LEGACY:
            legacy_compare(clip, cond, chunks, feats, pred, ap, fp, idx, gt)
        print(f"[{ci + 1}/{len(CLIPS)}] {clip:<20} {cond:<9} J={j:.4f} F={f:.4f} J&F={(j + f) / 2:.4f} "
              f"objects={res['n_objects']} frames={len(idx)} chunks={len(chunks)} {time.perf_counter() - t0:.1f}s")
        del feats, pred
    del chunks
    torch.cuda.empty_cache()
print(f"done in {time.perf_counter() - t_start:.0f}s | frame_index == range(T) for every clip: {all(frame_checks)}")

In [ ]:
# ── Cell 6 · Summary, paired statistics, save ──────────────────────────────
summary = {"per_condition": {}, "paired_anyup3d_vs": {}, "frame_index_ok": all(frame_checks)}
keys = ["J&F-Mean", "J-Mean", "J-Recall", "J-Decay", "F-Mean", "F-Recall", "F-Decay"]
print(f"{'condition':<10} " + " ".join(f"{k:>9}" for k in keys))
for cond in CONDITIONS:
    s = metrics.summarize_jf(PER_CLIP[cond])
    summary["per_condition"][cond] = s
    print(f"{cond:<10} " + " ".join(f"{s[k]:>9.4f}" for k in keys) + f"   objects={s['n_objects']}")

fp_ = lambda p: "—" if p is None else f"{p:.2e}"
if "anyup3d" in CONDITIONS:
    ours = summary["per_condition"]["anyup3d"]["per_clip"]
    for base in BASELINES:
        if base not in CONDITIONS:
            continue
        b = summary["per_condition"][base]["per_clip"]
        st = metrics.paired_stats([ours[c]["JF"] for c in CLIPS], [b[c]["JF"] for c in CLIPS])
        summary["paired_anyup3d_vs"][base] = st
        print(f"anyup3d vs {base:<9}: ΔJ&F={st['mean_delta']:+.4f} improved {st['n_improved']}/{st['n']} "
              f"sign p2={fp_(st['sign_p_two_sided'])} wilcoxon p2={fp_(st['wilcoxon_p_two_sided'])}")

if LEGACY:
    for cond in CONDITIONS:
        oj, of = np.mean(OLD_METRIC[cond]["J"]), np.mean(OLD_METRIC[cond]["F"])
        s = summary["per_condition"][cond]
        record(f"{cond}: old metric vs official (same masks)", None,
               f"old J={oj:.4f} F={of:.4f} J&F={(oj + of) / 2:.4f} | official J={s['J-Mean']:.4f} "
               f"F={s['F-Mean']:.4f} J&F={s['J&F-Mean']:.4f}")
    n_fail = sum(p is False for _, p, _ in RESULTS)
    print(f"\nLEGACY SUMMARY: {len(RESULTS)} checks, {n_fail} failed")
    for n, p, d in RESULTS:
        print(f"  [{'INFO' if p is None else ('PASS' if p else 'FAIL')}] {n}: {d}")
    summary["legacy_checks"] = [dict(name=n, passed=p, detail=d) for n, p, d in RESULTS]

config = dict(LEGACY=LEGACY, N_CLIPS=N_CLIPS, clips=CLIPS, CKPT_ANYUP3D=CKPT_ANYUP3D, SEED=SEED,
              CONDITIONS=CONDITIONS, backbone="swin_t", temporal_mode="per_frame", SWIN_NORM=SWIN_NORM,
              PROP_SIZE=PROP_SIZE, CHUNK_SIZE=CHUNK_SIZE, OVERLAP=OVERLAP, Q_CHUNK=Q_CHUNK, TAU=TAU, TOP_K=TOP_K,
              N_LAST=N_LAST, FRAME_STEP=FRAME_STEP if LEGACY else None, RUN_NAME=RUN_NAME,
              anyup3d_meta=UPS["anyup3d"]._extra_meta() if "anyup3d" in UPS else None,
              anyup2d_meta=UPS["anyup2d"]._extra_meta() if "anyup2d" in UPS else None,
              repo_branch=REPO_BRANCH)
csv_path, json_path = metrics.save_results(OUT_DIR, rows, summary, config, name="exp3")
zip_path = shutil.make_archive(f"{OUT_DIR}/masks", "zip", MASK_ROOT)
print("saved:", csv_path, json_path, zip_path)

In [ ]:
# ── Cell 7 · Optional: stills + comparison video ───────────────────────────
if RUN_VISUALS:
    PALETTE = np.array([[0, 0, 0], [255, 0, 0], [0, 255, 0], [0, 0, 255], [255, 255, 0], [0, 255, 255]], dtype=np.uint8)
    S = VISUAL_SIZE

    def overlay(frame_HWC, mask_HW, alpha=0.55):
        color = PALETTE[np.clip(mask_HW, 0, len(PALETTE) - 1)].astype(np.float32)
        fg = mask_HW > 0
        out = frame_HWC.copy().astype(np.float32)
        out[fg] = out[fg] * (1 - alpha) + color[fg] * alpha
        return np.clip(out, 0, 255).astype(np.uint8)

    def load_rgb(path):
        return np.array(Image.open(path).convert("RGB").resize((S, S), Image.BILINEAR))

    def load_ids(path):
        return cv2.resize(np.array(Image.open(path)), (S, S), interpolation=cv2.INTER_NEAREST).astype(np.int32)

    def panels(clip, i):
        fp, ap = davis.frame_paths(ROOT, clip), davis.ann_paths(ROOT, clip)
        frame = load_rgb(fp[i])
        out = [frame, overlay(frame, load_ids(ap[i]), alpha=0.6)]
        out += [overlay(frame, load_ids(Path(MASK_ROOT, c, clip, f"{fp[i].stem}.png"))) for c in CONDITIONS]
        return out

    def clip_idx(clip):
        n = len(davis.frame_paths(ROOT, clip))
        return list(range(0, n, FRAME_STEP)) if LEGACY else list(range(n))

    titles = ["Original", "GT"] + CONDITIONS
    vis = CLIPS[:N_VISUAL_CLIPS]
    fig, axes = plt.subplots(len(vis), len(titles), figsize=(4 * len(titles), 3.8 * len(vis)), squeeze=False)
    for r, clip in enumerate(vis):
        idx = clip_idx(clip)
        for c, (ax, p) in enumerate(zip(axes[r], panels(clip, idx[len(idx) // 2]))):   # middle frame
            ax.imshow(p); ax.axis("off")
            if r == 0:
                ax.set_title(titles[c], fontsize=11, fontweight="bold")
        axes[r][0].set_ylabel(clip, fontsize=9)
    plt.tight_layout()
    stills = f"{OUT_DIR}/side_by_side.png"
    plt.savefig(stills, dpi=150, bbox_inches="tight"); plt.show()

    clip = vis[0]
    video = f"{OUT_DIR}/{clip}_comparison.mp4"
    writer = cv2.VideoWriter(video, cv2.VideoWriter_fourcc(*"mp4v"), VISUAL_FPS, (S * len(titles), S))
    for i in clip_idx(clip):
        writer.write(cv2.cvtColor(np.concatenate(panels(clip, i), axis=1), cv2.COLOR_RGB2BGR))
    writer.release()
    print("saved:", stills, video)
else:
    print("RUN_VISUALS=False — skipped")